# M3 — Tiền Xử Lý, Gán Nhãn & Trích Xuất Đặc Trưng (Colab + Drive)

Chạy **sau** khi M1 (crawl arXiv + Semantic Scholar) và M2 (parsing TeX/BibTeX) đã hoàn thành.
Quy trình: GitHub branch → Drive → đọc dữ liệu M1/M2 → gán nhãn thủ công + tự động → trích xuất 14 đặc trưng → chia split → xuất Parquet.

Các bước:
1. **SET UP** — Clone code, cài thư viện, chạy tests.
2. **Mount Google Drive** — Kết nối Drive để đọc/ghi dữ liệu.
3. **Trỏ Drive** — Cấu hình đường dẫn, kiểm tra dữ liệu đầu vào, override `config`.
4. **Nạp dữ liệu** — Đọc `bibtex.jsonl` và `references.jsonl` từ M1/M2.
5. **Gán nhãn thủ công** — Xuất CSV mẫu cho 5 bài báo, nhóm kiểm tra và import lại.
6. **Chạy pipeline** — Gán nhãn tự động, kiểm tra quota, chia split, xuất Parquet.
7. **Trực quan hóa** — Vẽ biểu đồ phân bố 14 đặc trưng cho Poster A2.
8. **Lưu về Drive** — Copy Parquet + biểu đồ vào Drive.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M3-matching"                                         # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m3_matching.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive và cấu hình đường dẫn

Chỉnh `RANGE_ID` cho khớp với range bạn phụ trách. Notebook sẽ đọc dữ liệu M1/M2 đã có trên Drive và ghi kết quả M3 vào thư mục riêng.

Cấu trúc Google Drive kỳ vọng:

```text
MyDrive/
├── references/
│   └── range_1/           # Kết quả S2 từ M1
│       ├── reference/     # *.json từng bài
│       └── tables/
│           ├── papers.json
│           └── references.json
├── M2_processed/
│   └── range_1/           # Kết quả M2
│       └── tables/

│           └── bibtex.jsonl
└── M3_processed/
    └── range_1/           # Output notebook này (MY_M3_DIR)
        ├── tables/
        │   ├── bibtex.jsonl
        │   ├── references.jsonl
        │   ├── train_pairs.parquet
        │   ├── valid_pairs.parquet
        │   └── test_pairs.parquet
        └── labels/
            └── manual_templates/
```

In [ ]:
RANGE_ID = 1  # TODO: đổi thành range của bạn
K_MEMBERS = 1  # TODO: 1 nếu làm cá nhân, 2 nếu nhóm đôi (quota = 5*K bài, 100*K entries)


In [ ]:
import os
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# 1. Thư mục dữ liệu M1 (S2 references) — chỉ đọc
M1_REFS_DIR = Path(
    f"/content/drive/MyDrive/references/range_{RANGE_ID}"
)

# 2. File zip kết quả M2 — chỉ đọc (sẽ giải nén bibtex.jsonl)
M2_ZIP = Path(
    f"/content/drive/MyDrive/M2_processed/range_{RANGE_ID}/M2_range_{RANGE_ID}_tables.zip"
)

# 3. Thư mục output M3 — đọc/ghi
MY_M3_DIR = Path(
    f"/content/drive/MyDrive/M3_matching/range_{RANGE_ID}"
)
TABLES_DIR = MY_M3_DIR / "tables"
LABELS_DIR = MY_M3_DIR / "labels"
M2_EXTRACT_DIR = MY_M3_DIR / "m2_extracted"

for path in (MY_M3_DIR, TABLES_DIR, LABELS_DIR, M2_EXTRACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

# 4. Giải nén bibtex.jsonl từ zip M2 (chỉ file M3 cần, tránh giải nén 1.6 GB)
m2_bibtex_jsonl = M2_EXTRACT_DIR / "bibtex.jsonl"
if not m2_bibtex_jsonl.exists():
    assert M2_ZIP.exists(), (
        f"Chưa tìm thấy {M2_ZIP} "
        "— kiểm tra lại đường dẫn Drive (MyDrive/Shared with me) và kết quả M2"
    )
    with zipfile.ZipFile(M2_ZIP) as zf:
        names = set(zf.namelist())
        assert "bibtex.jsonl" in names, (
            f"Zip không có bibtex.jsonl; các file hiện có: {sorted(names)[:10]}"
        )
        zf.extract("bibtex.jsonl", M2_EXTRACT_DIR)
    print(f"✓ Đã giải nén bibtex.jsonl → {m2_bibtex_jsonl}")

M2_TABLES_DIR = M2_EXTRACT_DIR

# 5. Kiểm tra dữ liệu đầu vào từ M1/M2
m1_refs_json = M1_REFS_DIR / "tables" / "references.json"

assert m1_refs_json.exists(), (
    f"Chưa tìm thấy {m1_refs_json} "
    "— kiểm tra lại đường dẫn Drive và kết quả M1"
)
assert m2_bibtex_jsonl.exists(), (
    f"Chưa tìm thấy {m2_bibtex_jsonl} "
    "— kiểm tra lại đường dẫn Drive và kết quả M2"
)

# 6. Run ID
RUN_ID = os.environ.get(
    "M3_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

# 7. Override config và metrics để pipeline ghi đúng vào Drive
from src.common import config
import src.common.metrics as metrics_module

config.WORKING_DIR = MY_M3_DIR
config.TABLES_DIR  = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

print(f"Range          : {RANGE_ID}")
print(f"M1 refs từ     : {M1_REFS_DIR}")
print(f"M2 zip         : {M2_ZIP}")
print(f"M2 bibtex từ   : {m2_bibtex_jsonl}")
print(f"M3 output tại  : {MY_M3_DIR}")
print(f"TABLES_DIR     : {config.TABLES_DIR}")
print(f"LABELS_DIR     : {LABELS_DIR}")
print(f"RUN_ID         : {RUN_ID}")

## 2 — Nạp dữ liệu từ Drive (M1 References + M2 BibTeX)

Copy `bibtex.jsonl` từ M2 và `references.jsonl` từ M1 vào `TABLES_DIR` nội bộ để pipeline M3 đọc.
Dữ liệu gốc trên Drive **không bị sửa**.

In [ ]:
import json
import shutil
from pathlib import Path

from src.common.io import write_jsonl

# --- 1. Copy bibtex.jsonl từ M2 ---
dst_bib = TABLES_DIR / "bibtex.jsonl"
shutil.copy2(m2_bibtex_jsonl, dst_bib)
bib_lines = sum(1 for _ in open(dst_bib, encoding="utf-8"))
print(f"✓ bibtex.jsonl  : {bib_lines:,} dòng  ({dst_bib})")

# --- 2. Chuyển references.json (M1) → references.jsonl ---
dst_refs = TABLES_DIR / "references.jsonl"
with open(m1_refs_json, encoding="utf-8") as f:
    ref_records = json.load(f)
write_jsonl(dst_refs, ref_records)
print(f"✓ references.jsonl: {len(ref_records):,} dòng  ({dst_refs})")

print(f"\nDữ liệu đầu vào đã sẵn sàng tại: {TABLES_DIR}")

## 3 — Hỗ trợ gán nhãn thủ công

Tìm các bài báo có $\ge 20$ BibTeX entries, xuất CSV mẫu kèm gợi ý tự động để nhóm xác nhận.
Sau khi xuất, **mở CSV kiểm tra bằng mắt**, tự điền `groundtruth_arxiv_id` (hoặc để trống nếu không khớp) và đổi `is_verified` thành `YES` trước khi import.
`export_manual_template` không tự điền ground truth; cột `suggested_*` chỉ để tham khảo.

Yêu cầu tối thiểu: **5 × K_MEMBERS bài báo**, mỗi bài $\ge 20$ entries, tổng $\ge 100 × K_MEMBERS$ entries.


In [ ]:
import pandas as pd
from src.m3_matching.manual_tool import (
    export_manual_template,
    find_candidate_manual_papers,
    import_manual_csv,
    summarize_team_labels,
)

# 1. Tìm bài đủ điều kiện
candidates = find_candidate_manual_papers(
    bibtex_path=TABLES_DIR / "bibtex.jsonl",
    references_path=TABLES_DIR / "references.jsonl",
)
print(f"Tìm thấy {len(candidates)} bài có ≥ 20 BibTeX entries và có references:")
for pid, b_cnt, r_cnt in candidates[:10]:
    print(f"  - {pid}: {b_cnt} BibTeX | {r_cnt} References")

N_MANUAL = 5 * K_MEMBERS
assert len(candidates) >= N_MANUAL, (
    f"Chỉ có {len(candidates)} bài đủ điều kiện (cần ≥ {N_MANUAL}). "
    "Kiểm tra lại dữ liệu M1/M2."
)

# 2. Xuất CSV mẫu cho 5*K_MEMBERS bài đầu tiên
TEMPLATE_DIR = LABELS_DIR / "manual_templates"
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
selected_5 = [pid for pid, _, _ in candidates[:N_MANUAL]]

for pid in selected_5:
    out_csv = TEMPLATE_DIR / f"{pid}.csv"
    export_manual_template(
        pid, out_csv,
        bibtex_path=TABLES_DIR / "bibtex.jsonl",
        references_path=TABLES_DIR / "references.jsonl",
    )
    print(f"  -> Đã xuất template: {out_csv}")

print(f"\n✓ Đã xuất {len(selected_5)} file CSV mẫu tại: {TEMPLATE_DIR}")

# 3. Xem trước bảng mẫu đầu tiên
demo_pid = selected_5[0]
df_preview = pd.read_csv(TEMPLATE_DIR / f"{demo_pid}.csv")
display(df_preview[[
    "citation_key", "bib_title",
    "suggested_ref_title", "suggested_score",
    "groundtruth_arxiv_id", "is_verified",
]].head(10))


In [ ]:
# Chạy cell này SAU KHI nhóm đã kiểm tra bằng mắt, điền groundtruth và đổi is_verified=YES.
# Mỗi thành viên import file của mình; cell có thể chạy nhiều lần.

for pid in selected_5:
    csv_path = TEMPLATE_DIR / f"{pid}.csv"
    if not csv_path.exists():
        print(f"  [skip] Chưa có file: {csv_path}")
        continue
    try:
        out = import_manual_csv(csv_path, pid, labels_dir=LABELS_DIR)
        print(f"  ✓ Đã import nhãn thủ công {pid} → {out}")
    except ValueError as exc:
        print(f"  [skip] {pid}: {exc}")

summary = summarize_team_labels(labels_dir=LABELS_DIR, k=K_MEMBERS)
print("\n=== TIẾN ĐỘ GÁN NHÃN ===")
print(f"  Bài manual   : {summary['n_manual_papers']} (yêu cầu ≥ {5*K_MEMBERS})")
print(f"  Entries manual: {summary['n_manual_entries']} (yêu cầu ≥ {100*K_MEMBERS})")
print(f"  Quota status : {summary['quota_status']}")


## 4 — Chạy trọn gói pipeline M3

Thực hiện theo thứ tự:
- Gán nhãn tự động cho $\ge 10\%$ bài ngoài nhóm manual (`labelling.py`)
- Kiểm tra quota (`validate_label_quota`)
- Chia tập `train` / `valid` / `test` theo bài báo (`split.py`)
- Sinh bảng cặp ứng viên kèm 14 đặc trưng và nhãn 0/1 (`dataset.py`)
- Xuất 4 file Parquet vào `TABLES_DIR`

`MAX_AUTO` giới hạn số bài auto để tránh dataset quá lớn (khớp lệnh `--max-auto 450` trong `MATCHING_M3.md`).
Nếu `TABLES_DIR` có `elements.jsonl` (M2), `cite_count` được đếm từ các lệnh `\cite{...}`; nếu không, cột này bằng 0.


In [ ]:
from src.m3_matching.prepare_dataset import run_preparation_pipeline

RANDOM_SEED = 42
MAX_AUTO = 450  # 0 = gán tất cả bài auto

summary = run_preparation_pipeline(
    tables_dir=TABLES_DIR,
    labels_dir=LABELS_DIR,
    k=K_MEMBERS,
    seed=RANDOM_SEED,
    demo=False,
    max_auto=MAX_AUTO,
)

print("\n=================== KẾT QUẢ XUẤT BẢN DỮ LIỆU ===================")
print(f"  Số đặc trưng  : {len(summary['feature_names'])} features")
print(f"  Train : {summary['splits']['train']['n_pairs']:,} cặp "
      f"({summary['splits']['train']['n_positives']} pos, "
      f"{summary['splits']['train']['n_negatives']} neg)")
print(f"  Valid : {summary['splits']['valid']['n_pairs']:,} cặp "
      f"({summary['splits']['valid']['n_positives']} pos, "
      f"{summary['splits']['valid']['n_negatives']} neg)")
print(f"  Test  : {summary['splits']['test']['n_pairs']:,} cặp "
      f"({summary['splits']['test']['n_positives']} pos, "
      f"{summary['splits']['test']['n_negatives']} neg)")
print("==================================================================")

print("\nCác file đã xuất:")
for f in summary["exported_files"]:
    p = Path(f)
    if p.exists():
        print(f"  ✓ {p.name}: {p.stat().st_size:,} bytes")
    else:
        print(f"  ✗ {p.name}: MISSING")


## 5 — Trực quan hóa 14 đặc trưng (cho Poster A2)

Vẽ biểu đồ phân bố 3 đặc trưng đại diện giữa nhãn 0 và nhãn 1 trên tập train.
File ảnh được lưu vào Drive cùng với Parquet.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

train_df = pd.read_parquet(TABLES_DIR / "train_pairs.parquet")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Phân bố độ tương đồng tiêu đề (title_token_sort)
sns.histplot(
    data=train_df, x="title_token_sort", hue="label",
    bins=30, kde=True, ax=axes[0], palette={0: "#e74c3c", 1: "#2ecc71"},
    common_norm=False, stat="density",
)
axes[0].set_title("Phân bố độ tương đồng Tiêu đề (Token Sort Ratio)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Score (0.0 .. 1.0)")
axes[0].legend(title="Nhãn", labels=["Trùng khớp (1)", "Khác bài (0)"])

# 2. Phân bố trùng khớp họ tác giả (author_jaccard)
sns.histplot(
    data=train_df, x="author_jaccard", hue="label",
    bins=20, kde=False, ax=axes[1], palette={0: "#e74c3c", 1: "#2ecc71"},
    common_norm=False, stat="density",
)
axes[1].set_title("Phân bố trùng khớp Họ Tác giả (Jaccard)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Jaccard Score")
axes[1].legend(title="Nhãn", labels=["Trùng khớp (1)", "Khác bài (0)"])

# 3. Phân bố chênh lệch năm công bố (year_diff)
year_df = train_df[train_df["year_diff"] < 20]
sns.boxplot(
    data=year_df, x="label", y="year_diff", ax=axes[2],
    palette={0: "#e74c3c", 1: "#2ecc71"},
)
axes[2].set_title("Độ lệch Năm Công bố theo Nhãn", fontsize=12, fontweight="bold")
axes[2].set_xticklabels(["Khác bài (0)", "Trùng khớp (1)"])
axes[2].set_ylabel("Chênh lệch năm tuyệt đối")

plt.tight_layout()
poster_fig_path = TABLES_DIR / "poster_feature_distribution.png"
plt.savefig(poster_fig_path, dpi=300)
plt.show()
print(f"✓ Đã lưu biểu đồ: {poster_fig_path}")

## 6 — Lưu kết quả về Google Drive

Copy toàn bộ file Parquet và ảnh biểu đồ từ `TABLES_DIR` vào Drive.
Vì `TABLES_DIR` đã **là thư mục Drive** (`MY_M3_DIR/tables/`), dữ liệu đã được lưu trực tiếp trong quá trình pipeline. Cell này chỉ kiểm tra và in xác nhận.

In [ ]:
export_files = [
    "bibtex.jsonl",
    "references.jsonl",
    "train_pairs.parquet",
    "valid_pairs.parquet",
    "test_pairs.parquet",
    "dataset_pairs.parquet",
    "poster_feature_distribution.png",
]

print(f"Kiểm tra kết quả trong: {TABLES_DIR}")
print(f"Run ID: {RUN_ID}")
print()
all_ok = True
for fname in export_files:
    p = TABLES_DIR / fname
    if p.exists():
        print(f"  ✓ {fname}: {p.stat().st_size:,} bytes")
    else:
        print(f"  ✗ {fname}: MISSING")
        all_ok = False

print()
if all_ok:
    print("HOÀN TẤT TOÀN BỘ QUY TRÌNH MILESTONE 3!")
else:
    print("[warn] Một số file bị thiếu — kiểm tra lại các bước trên.")

## Sau khi chạy

Gửi cho người xử lý tiếp theo (huấn luyện mô hình):
- `tables/bibtex.jsonl`
- `tables/references.jsonl`
- `tables/train_pairs.parquet`
- `tables/valid_pairs.parquet`
- `tables/test_pairs.parquet`
- `tables/dataset_pairs.parquet`

Toàn bộ nằm trong `MyDrive/M3_processed/range_{RANGE_ID}/tables/`.

## Tùy chọn: Upload code M3 lên Drive

Nếu muốn lưu toàn bộ code Python của M3 lên Drive để share với cả nhóm, chạy cell sau.

In [ ]:
import shutil
from pathlib import Path

# Tạo folder M3_code_backup trên Drive
m3_backup_dir = Path("/content/drive/MyDrive/M3_code_backup")
m3_backup_dir.mkdir(parents=True, exist_ok=True)

# Copy toàn bộ Python files từ project/src/m3_matching
source_m3 = Path("/content/project/src/m3_matching")
target_m3 = m3_backup_dir / "m3_matching"

if target_m3.exists():
    shutil.rmtree(target_m3)
shutil.copytree(source_m3, target_m3, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".pytest_cache"))

# Đếm files
files = list(target_m3.glob("*.py")) + list(target_m3.glob("*.ipynb")) + list(target_m3.glob("*.md"))
print(f"✓ Đã backup {len(files)} files M3 code lên Drive:")
print(f"  {m3_backup_dir}")
for f in sorted(files):
    print(f"  - {f.name}")